About the Dataset:

1. id: unique id for a news article
2. title: the title of a news article
3. author: author of the news article
4. text: the text of the article; could be incomplete
5. label: a label that marks whether the news article is real or fake:
           0: Fake news
           1: real News





Importing the Dependencies

In [2]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# printing the stopwords in English
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Pre-processing

In [12]:
import kagglehub
import os

dataset_id = "saurabhshahane/fake-news-classification"
# kagglehub.dataset_download returns the path to the root directory of the downloaded dataset
downloaded_dataset_root = kagglehub.dataset_download(dataset_id)

# Find the 'WELFake_Dataset.csv' file within the downloaded directory
# This handles cases where the dataset might be in a subdirectory or direct
path = None
for root, _, files in os.walk(downloaded_dataset_root):
    if 'WELFake_Dataset.csv' in files:
        path = os.path.join(root, 'WELFake_Dataset.csv')
        break

if path is None:
    raise FileNotFoundError(f"WELFake_Dataset.csv not found in {downloaded_dataset_root} or its subdirectories.")

Using Colab cache for faster access to the 'fake-news-classification' dataset.


In [20]:
# loading the dataset to a pandas DataFrame
news_dataset = pd.read_csv(path)

In [14]:
news_dataset.shape

(72134, 4)

In [15]:
# print the first 5 rows of the dataframe
news_dataset.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [16]:
# counting the number of missing values in the dataset
news_dataset.isnull().sum()

,0
Unnamed: 0,0
title,558
text,39
label,0


In [17]:
# replacing the null values with empty string
news_dataset = news_dataset.fillna('')

In [21]:
# separating the data & label
X = news_dataset.drop(columns={'label','Unnamed: 0'}, axis=1)
Y = news_dataset['label']

In [22]:
print(X.head())
print(Y.head())

                                               title  \
0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1                                                NaN   
2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3  Bobby Jindal, raised Hindu, uses story of Chri...   
4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  
0  No comment is expected from Barack Obama Membe...  
1     Did they post their votes for Hillary already?  
2   Now, most of the demonstrators gathered last ...  
3  A dozen politically active pastors came here f...  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...  
0    1
1    1
2    1
3    0
4    1
Name: label, dtype: int64


Stemming:

Stemming is the process of reducing a word to its Root word

example:
actor, actress, acting --> act

In [28]:
# combining the title and text columns
news_dataset['content'] = news_dataset['title'].astype(str) + ' ' + news_dataset['text'].astype(str)
port_stem = PorterStemmer()

In [29]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [31]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [32]:
#separating the data and label
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [33]:
print(X)

['law enforc high alert follow threat cop white blacklivesmatt fyf terrorist video comment expect barack obama member fyf fukyoflag blacklivesmatt movement call lynch hang white peopl cop encourag other radio show tuesday night turn tide kill white peopl cop send messag kill black peopl america one f yoflag organ call sunshin radio blog show host texa call sunshin f ing opinion radio show snapshot fyf lolatwhitefear twitter page p show urg support call fyf tonight continu dismantl illus white snapshot twitter radio call invit fyf radio show air p eastern standard time show caller clearli call lynch kill white peopl minut clip radio show heard provid breitbart texa someon would like refer hannib alreadi receiv death threat result interrupt fyf confer call unidentifi black man said mother f ker start f ing like us bunch ni er takin one us roll said caus alreadi roll gang anyway six seven black mother f cker see white person lynch ass let turn tabl conspir cop start lose peopl state emerg

In [34]:
print(Y)

[1 1 1 ... 0 0 1]


In [35]:
Y.shape

(72134,)

In [36]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [37]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 13657264 stored elements and shape (72134, 162203)>
  Coords	Values
  (0, 938)	0.019104619426517897
  (0, 1282)	0.017363778513914716
  (0, 2131)	0.052457780993620334
  (0, 2783)	0.020231302394732035
  (0, 3614)	0.029904475345647965
  (0, 3999)	0.02747310458208844
  (0, 4264)	0.023865946576073604
  (0, 4335)	0.05055646943154232
  (0, 4846)	0.01513932938772633
  (0, 4862)	0.02486341752399553
  (0, 6013)	0.014596932940161228
  (0, 6507)	0.057303304534120254
  (0, 6845)	0.01589099748716943
  (0, 8437)	0.12657603668480968
  (0, 8976)	0.015516676506767142
  (0, 10478)	0.06692816334064453
  (0, 11430)	0.018962618491219916
  (0, 12727)	0.01580162854760987
  (0, 14072)	0.018345912143817013
  (0, 14679)	0.01785037970922704
  (0, 15442)	0.19279395985841352
  (0, 15499)	0.08125624068348719
  (0, 15611)	0.0888918729364855
  (0, 15886)	0.029332963149593518
  (0, 18063)	0.10843561013885229
  :	:
  (72133, 132638)	0.031715743461707
  (72133

Splitting the dataset to training & test data

In [38]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

Training the Model: Logistic Regression

In [39]:
model = LogisticRegression()

In [40]:
model.fit(X_train, Y_train)

LogisticRegression()

Evaluation

accuracy score

In [41]:
# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [42]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9636265964267766


In [43]:
# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [44]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9486379704720316


Making a Predictive System

In [45]:
X_new = X_test[3]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('The news is Real')
else:
  print('The news is Fake')

[0]
The news is Real


In [46]:
print(Y_test[3])

0
